In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging

import matplotlib.pyplot as plt  # type: ignore
import numpy as np

from mlpng import Core
from mlpng.utils import (
    setup_logging,
    plot_cl_alm,
    plot_predictions,
    plot_histogram,
    plot_elsner_comp,
    # pol_str,
)
from mlpng.generator import generate_alm, generate_alm_nl, lens_alms
from mlpng.utils.utils import print_errors

logger = setup_logging(__name__, level=logging.DEBUG)

mpi_comm = None

## Setup

In [ ]:
core = Core(
    [
        "settings/elsner.json",
        "--nsims",
        "33",
        "--narray",
        "1",
        "--fnl_range",
        "-100",
        "100",
        "--pols",
        "T",
        # "--no-noise",
        # "--nside",
        # "128",
        # "--lmax",
        # "383",
    ]
)

In [ ]:
core.init_estimator(verbose=False)

## Alm

This code generates the alms

$$a_{\ell m} = a_{\ell m}^{{G}} + f_{NL}^X a_{\ell m}^{NG}$$
with
$$a_{\ell m}^{NG,loc'} = \int dr r^2 \left[ \alpha_\ell(r)\left(\int d^2 \hat{n} Y_{\ell m}^\star (\hat{n}) B(r,\hat{n})^2 \right)\right]$$
and
$$\alpha_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^2 \Delta_\ell^T(k) j_\ell(k r)$$
$$\beta_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^{-1} \Delta_\phi \Delta_\ell^T(k) j_\ell(k r)$$
$$B(r, \hat{n}) = \sum_{\ell,m} \frac{\beta_\ell (r)}{C_\ell} a_{\ell m} Y_{\ell m}$$
where $\Delta_\phi$ is primordial normalization, $\Delta_\ell^T(k)$ is the transfer function, $j_\ell(k r)$ are the spherical bessel functions   

In [ ]:
alm_l = generate_alm(core)
print(alm_l[:, :, :2])
alm_l[:, :, :2] = 0
print(alm_l[:, :, 96])
alm_l[:, :, 96] = 0
alm_ng = generate_alm_nl(core, alm_l)

In [ ]:
fnls = core.rng.uniform(core.fnl_min, core.fnl_max, (core.nsims, core.ndups, 1, 1))
alms = alm_l[:, None, core.pol_idxs()] + fnls * alm_ng[:, None]

In [ ]:
plot_cl_alm(
    core,
    alm_l[0],
    title="linear alms",
    # labels=plt_labels,
    plot_camb=True,
    plot_noise=True,
    plot_full_camb=False,
    show=True,
)

plot_cl_alm(
    core,
    alm_ng[0],
    title="non-linear alms",
    plot_camb=False,
    plot_noise=True,
    plot_full_camb=False,
    show=True,
)

In [ ]:
plot_cl_alm(
    core,
    alms[0, 0],
    title="alms final",
    plot_camb=True,
    plot_noise=True,
    plot_full_camb=True,
    show=True,
)

In [ ]:
sim = core.rng.integers(core.nsims)
eidx = core.rng.integers(1, 1001)
plot_elsner_comp(
    core,
    alm_l[sim, core.pol_idxs()],
    alm_ng[sim],
    index=eidx,
    show=True,
    plot_func=plt.semilogy,
)
plot_elsner_comp(
    core,
    alm_l[sim, core.pol_idxs()],
    alm_ng[sim],
    index=eidx,
    show=True,
    plot_func=plt.plot,
)
plot_elsner_comp(
    core,
    alm_l[sim, core.pol_idxs()],
    alm_ng[sim],
    index=eidx,
    show=True,
    plot_func=plt.loglog,
)

In [ ]:
alm_l_avg = np.mean(alm_l[:, core.pol_idxs()], axis=0)
alm_ng_avg = np.mean(alm_ng, axis=0)

plot_elsner_comp(
    core,
    alm_l_avg,
    alm_ng_avg,
    average=core.nsims,
    title="avged comp",
    show=True,
    plot_func=plt.semilogy,
)

## Estimator

In [ ]:
# pols = core.pol_idxs()

# ib = np.zeros_like(core.c_ell)
# ib[pols, core.lmin :] = 1 / core.b_ell[pols, core.lmin :]

# cov = core.b_ell**2 * core.c_ell + core.n_ell
# # cov = core.c_ell + ib * core.n_ell * ib

# ic_ell = np.zeros_like(cov)
# ic_ell[pols, core.lmin :] = 1 / cov[pols, core.lmin :]
# ic_ell = ic_ell[pols]
# # ic_ell *= core.b_ell[pols] ** 2

# fisher = core.estimator.compute_fisher_isotropic(ic_ell, comm=None)
# print(f"Fisher: {fisher}, standard deviation: {1 / np.sqrt(fisher)}")

In [ ]:
# from mlpng.core import get_itotcov_ell

# pol = core.pol_idxs()

# # cov = core.b_ell**2 * core.c_ell + core.n_ell
# cov = core.c_ell
# cov = cov[pols]

# f_ic = np.zeros_like(cov)
# f_ic[..., core.lmin :] = 1 / cov[..., core.lmin :]

# inoise = np.full(cov.shape, 1e-16)
# inoise[..., core.lmin :] = 1 / core.n_ell[pols, core.lmin :]

# f_ic = get_itotcov_ell(f_ic, inoise, core.b_ell[pols])

# logger.debug("Computing Fisher, %s", f_ic.shape)
# fisher = core.estimator.compute_fisher_isotropic(f_ic[pol, pol], comm=mpi_comm)
# print(f"Fisher: {fisher}, standard deviation: {1 / np.sqrt(fisher)}")

In [ ]:
fisher = core.estimator.compute_fisher_isotropic(core.icov, comm=mpi_comm)
print(f"Fisher: {fisher}, standard deviation: {1 / np.sqrt(fisher)}")

In [ ]:
estimates, _, _, _ = core.estimator.compute_estimate_batch(
    lambda idx: core.icov_func(alms[idx, 0]),
    range(core.nsims),
    theta_batch=int(np.floor(1.5 * core.lmax + 1)) // core.n_cpus,
    fisher=fisher,
    lin_term=0,
)

In [ ]:
print("fnl shape", fnls.shape, "estimates shape", estimates.shape, "fisher", fisher)
fnls_flat = fnls[:, 0, 0, 0]
print_errors(fnls_flat, estimates, fisher)
plot_predictions(fnls_flat, estimates, fisher=fisher, show=True)
plot_histogram(fnls_flat, estimates, show=True)

## Lensing

In [ ]:
alm_lensed, phi_map = lens_alms(core, alms)

In [ ]:
sim = 0
for i, pol in enumerate(core.pol_idxs()):
    pstr = core.pols[pol]

    ylabel = r"$\ell(\ell+1)/2\pi\;C_{\ell}" + f"^{pstr}$"
    plot_cl_alm(
        core,
        alm_lensed[sim, i],
        # save_file=filebase,
        ylabel=ylabel,
        plot_camb=True,
        plot_noise=False,
        plot_full_camb=True,
        show=True,
        lmin=2,
    )

In [ ]:
lens_estimates, _, _, _ = core.estimator.compute_estimate_batch(
    lambda a: core.icov_func(alm_lensed[a, 0, core.pol_idxs(pretrimmed=True)]),
    range(core.nsims),
    theta_batch=int(np.floor(1.5 * core.lmax + 1)) // core.n_cpus,
    fisher=fisher,
    lin_term=0,
)

In [ ]:
print(
    "fnl shape", fnls.shape, "estimates shape", lens_estimates.shape, "fisher", fisher
)
fnls_flat = fnls[:, 0, 0, 0]
print_errors(fnls_flat, lens_estimates, fisher)
plot_predictions(fnls_flat, lens_estimates, fisher=fisher, show=True)
plot_histogram(fnls_flat, lens_estimates, show=True)